# 総計4,536ヵ所の源泉を眺める日本全国「温泉の旅」―データ.可視化で楽しむ「温泉の科学」入門編


## 記事掲載用

In [ ]:
import numpy as np; import pandas as pd; import folium; import matplotlib.pyplot as plt; import math; 
from matplotlib.colors import rgb2hex; plt.rcParams.update({"font.family":"Hiragino Sans","font.size":24})


REPLACE_RULES = [ # 置換ルール（順序厳守）
    ("III", "3"), ("II", "2"),("Fe(2)", "鉄(Ⅱ)"), ("Fe(2,3)", "鉄(Ⅱ+Ⅲ)"),
    ("Fe(2+3)", "鉄(Ⅱ+Ⅲ)"),
    ("H_2S", "硫化水素"), ("H_2SiO_3", "メタケイ酸"),("HCO_3", "炭酸水素塩"), ("HCO3", "炭酸水素塩"),
    ("H2S", "硫化水素"),
    ("HO_3", "炭酸水素塩"), ("HC0_3", "炭酸水素塩"), ("CO_3", "炭酸水素塩"),  # 誤植対応
    ("CO_2", "二酸化炭素"), ("CO2", "二酸化炭素"),
    ("SO_4", "硫酸塩"), ("SO4", "硫酸塩"),
    ("Fe", "鉄"), ("Na", "ナトリウム"), ("Mg", "マグネシウム"), ("Rn", "ラドン"),
    ("Br", "臭素"), ("Al", "アルミニウム"), ("As", "ヒ素"), ("Cl", "塩化物"),
    ("Cu", "銅"), ("Ca", "カルシウム"), ("F", "フッ素"),
    ("Ｓ", "硫黄"), ("S", "硫黄"), ("B", "ホウ酸"), ("I", "ヨウ素"), ("K", "カリウム"),
    ("、", ","), ('"', ""),
    ('（', "("), ('）', ")"),
    ('(・', "・"), ('・)', "・"), ('・(', "・"), (')・', "・"),
    ('-()', "-"), (')-', "-"),
    ('泉(', "泉・"), ('塩(', "塩・"),
    ('塩化物(硫化水素型', "硫化水素型塩化物"),
    ('塩化物(カルシウム', "カルシウム塩化物"),('炭酸水素塩-', "炭酸水素塩"),
    ('(Ⅱ-', "(Ⅱ)-"), ('(Ⅲ-', "(Ⅲ)-"),('(Ⅱ・', "(Ⅱ)・"), ('(Ⅲ・', "(Ⅲ)・"),]
ALIASES = { "名称": ["名称"], "Name": ["Name"], "市町村": ["市町村"], "略記泉質": ["略記泉質"],
    "深度(m)": ["深度(m)"], "湧出量": ["湧出量"], "温度上限": ["温度上限"], "pH上限": ["pH上限"],}

def _to_float(s: pd.Series) -> pd.Series:
    return pd.to_numeric(s.astype("string").replace("-", pd.NA), errors="coerce").astype(float)

def _to_str(s: pd.Series) -> pd.Series:
    x = s.astype("string").replace("-", pd.NA)                          # '-'→NaN
    return x.replace(r"^\s*$", pd.NA, regex=True)                       # 空白のみ→NaN

def normalize_tokens(short_str) -> list[str]:
    if short_str is None or pd.isna(short_str): return []
    raw = str(short_str)
    for src, dst in REPLACE_RULES: raw = raw.replace(src, dst)          # 置換（順序厳守）
    toks = [t.strip() for t in raw.split(",") if t and t.strip() not in {"", "-"}]  # ','分割＋空除外
    out = []
    for t in toks: # 先頭/末尾の括弧を軽く正規化
        t = t[1:] if t.startswith("(") else t                           # 先頭 '(' を落とす
        t = t[:-1] if t.endswith(")") else t                            # 末尾 ')' を落とす
        t = (t[:-2] + ")") if t.endswith(")-") else t                   # ')-' を救済
        out.append(t)
    return out

def load_onsen_tsv(path: str, encoding: str = "cp932") -> pd.DataFrame:
    _find_col = lambda df, cands: next((c for c in cands if c in df.columns), None)
    df = pd.read_csv(path, sep="\t", header=0, dtype="string", encoding=encoding, na_filter=False)
    if df.shape[1] < 6: raise ValueError(f"少なくとも6列（経度DMS3列＋緯度DMS3列）が必要です。列数={df.shape[1]}")
    lon = sum(pd.to_numeric(df.iloc[:, i], errors="coerce") / d for i, d in zip([0,1,2],[1,60,3600]))
    lat = sum(pd.to_numeric(df.iloc[:, i], errors="coerce") / d for i, d in zip([3,4,5],[1,60,3600]))
    df["経度"], df["緯度"] = lon.astype(float), lat.astype(float)
    for logical in ["名称", "Name", "市町村", "略記泉質"]: # string列 / float列 を「logical名」に寄せる（無ければ作る）
        col = _find_col(df, ALIASES.get(logical, [logical]))
        df[logical] = _to_str(df[col]) if col else pd.Series([pd.NA]*len(df), dtype="string")
    for logical in ["深度(m)", "湧出量", "温度上限", "pH上限"]:
        col = _find_col(df, ALIASES.get(logical, [logical]))
        df[logical] = _to_float(df[col]) if col else np.nan
    df["市町村 名称"] = (df["市町村"].fillna("") + " " + df["名称"].fillna("")).str.strip()
    df["泉質"] = df["略記泉質"].apply(normalize_tokens); name = df["名称"].astype("string")
    ok = name.notna() & (name.str.strip() != "") & (name.str.strip() != "-") & df["緯度"].notna() & df["経度"].notna()
    return df[ok].reset_index(drop=True)

def make_popup_html(row: pd.Series) -> str:
    g = lambda k: "" if (k not in row or pd.isna(row[k])) else str(row[k])
    senshitsu = row.get("泉質", [])
    if not isinstance(senshitsu, list): senshitsu = []
    lines = [f"<b>{g('市町村 名称')}</b>",f"Name: {g('Name')}",f"泉質: {', '.join(senshitsu)}",
        f"深度(m): {g('深度(m)')}",f"湧出量: {g('湧出量')}",f"温度上限: {g('温度上限')}",
        f"pH上限: {g('pH上限')}",f"緯度: {g('緯度')}, 経度: {g('経度')}",]
    return "<br>".join([ln for ln in lines if not ln.endswith(": ") and not ln.endswith(":")])

def val2color(v, vmin, vmax, cmap):
    if v is None or (isinstance(v, float) and math.isnan(v)): return "#999999"
    if vmax == vmin: return rgb2hex(cmap(0.5)[:3])
    return rgb2hex(cmap(min(1.0, max(0.0, (float(v) - vmin) / (vmax - vmin) )))[:3])

def val2alpha(v, vmin, vmax, a_min, a_max): # vを[a_min, a_max]へ線形マップ（欠損はa_min）
    if v is None or (isinstance(v, float) and math.isnan(v)): return float(a_min)
    if vmax == vmin: return float((a_min + a_max) / 2)
    t = (float(v) - vmin) / (vmax - vmin); t = min(1.0, max(0.0, t))
    return float(a_min + (a_max - a_min) * t)

def build_folium_map_token_layers(
    df: pd.DataFrame, start_zoom: int = 6, tiles: str = "OpenStreetMap",
    color_col: str = "湧出量",          # 色/透明度の元にする列
    cmap=plt.cm.cool,                  # 値→色 のカラーマップ
    cmin: float | None = 0.0, cmax: float | None = 5000.0, # 値の下限と上限
    radius_m: float = 400, weight: int = 5, fill_opacity: float = 0.3, opacity: float = 1.0,
    map_width: int = 900, map_height: int = 900,
    mode: str = "value_color",         # "value_color" | "solid" | "color_alpha"
    solid_color: str = "#FF0000",      # mode="solid" のときの固定色
    alpha_col: str | None = None,      # mode="color_alpha" で透明度に使う列（Noneならcolor_col）
    alpha_min: float = 0.05,alpha_max: float = 0.6, # 透明度の下限＆上限
):
    dfx = df.dropna(subset=["緯度", "経度"]).copy()
    if dfx.empty: raise ValueError("緯度・経度が有効な行がありません。DMS列や欠損を確認してください。")
    center = [float(dfx["緯度"].mean()), float(dfx["経度"].mean())]
    m = folium.Map(location=center, zoom_start=start_zoom, tiles=tiles, width=map_width)
    m.get_root().height = map_height
    s = pd.to_numeric(dfx.get(color_col), errors="coerce") # 値スケール（色用）
    vmin = float(s.min()) if cmin is None else float(cmin)
    vmax = float(s.max()) if cmax is None else float(cmax)
    if mode == "color_alpha":    # 透明度スケール
        ac = alpha_col or color_col
        sa = pd.to_numeric(dfx.get(ac), errors="coerce")
        amin = float(sa.min()) if cmin is None else float(cmin)
        amax = float(sa.max()) if cmax is None else float(cmax)
    for _, row in dfx.iterrows():
        lat, lon = float(row["緯度"]), float(row["経度"])
        popup, tooltip = make_popup_html(row), row.get("市町村 名称", "")
        if mode == "solid":
            color = solid_color; fo = fill_opacity # 全部同じ色で透明度は固定
        elif mode == "color_alpha":
            v = row.get(color_col, np.nan); color = val2color(v, vmin, vmax, cmap) # 値→色
            v_a = row.get(alpha_col or color_col, np.nan)
            fo = val2alpha(v_a, amin, amax,alpha_min,alpha_max); fill_opacity = fo # 値→透明度
        folium.Circle(
            location=[lat, lon], popup=popup, tooltip=tooltip, radius=radius_m,
            color=color,weight=weight,fill=True,fill_color=color,fill_opacity=fo,opacity=fill_opacity
        ).add_to(m)
    return m

df = load_onsen_tsv("ONSEN.TXT", encoding="cp932")
build_folium_map_token_layers(df, mode="solid", solid_color="#FF0000")

In [ ]:
build_folium_map_token_layers(df, mode="color_alpha", color_col="湧出量", cmap=plt.cm.bwr, cmin=0, cmax=750,)

In [ ]:
def scatter_series( s: pd.Series, *, x=None, annot=None, annot_fmt=None, annot_offset=(2,2), 
    annot_fontsize=16,c=None, cmap=plt.cm.viridis, cmin=None, cmax=None, colorbar=True, c_label=None,
    title=None, xlabel=None, ylabel=None, marker="o", size=20, alpha=0.9, figsize=(14,14),
    xlim=None, ylim=None, grid=True,):
    if not isinstance(s, pd.Series): raise TypeError("s must be a pandas.Series")
    x = s.index if x is None else (x.values if isinstance(x, pd.Series) else x)          # x未指定→index
    if len(x) != len(s): raise ValueError(f"len(x)={len(x)} must match len(s)={len(s)}")
    y = s.values                                                                         # yはSeries値
    labx = xlabel or (getattr(s.index,"name",None) or getattr(x,"name",None) or "x")     # xラベル自動
    laby = ylabel or (s.name or "value")                                                 # yラベル自動
    title = title or f"Scatter: {laby}"
    if annot is None and annot_fmt is not None: annot = [annot_fmt.format(v) for v in y]
    if isinstance(annot, pd.Series): annot = annot.values
    if annot is not None:
        if len(annot) != len(s): raise ValueError(f"len(annot)={len(annot)} must match len(s)={len(s)}")
        annot = ["" if pd.isna(a) else str(a) for a in annot]                            # NaN→空
    c_label = c_label or (getattr(c, "name", None) or "c")                               # カラーバーラベル
    if c is not None:
        c = (c.values if isinstance(c, pd.Series) else c)
        if len(c) != len(s): raise ValueError(f"len(c)={len(c)} must match len(s)={len(s)}")
        c = pd.to_numeric(pd.Series(c), errors="coerce").values                          # 数値化（NaN可）
        cmin = float(pd.Series(c).min(skipna=True)) if cmin is None else float(cmin)     # vmin
        cmax = float(pd.Series(c).max(skipna=True)) if cmax is None else float(cmax)     # vmax
    plt.figure(figsize=figsize)
    sc = plt.scatter(x, y, marker=marker, s=size, alpha=alpha, c=c, cmap=cmap, vmin=cmin, vmax=cmax)
    plt.title(title); plt.xlabel(labx); plt.ylabel(laby)                                  # 見出し
    if xlim: plt.xlim(*xlim)
    if ylim: plt.ylim(*ylim)                                                              # 軸範囲
    if c is not None and colorbar: plt.colorbar(sc).set_label(c_label)                    # カラーバー
    if annot is not None:
        for xv, yv, t in zip(x, y, annot):
            if t: plt.annotate(t, (xv,yv), textcoords="offset points", xytext=annot_offset,
                               ha="left", va="bottom", fontsize=annot_fontsize, alpha=alpha)
    if grid: plt.grid(True); plt.show()

scatter_series(df["温度上限"], x=df["深度(m)"], c=df["湧出量"],cmap=plt.cm.Spectral, cmin=0, cmax=1000,
    size=80, alpha=0.5, colorbar=True, c_label="湧出量", title="深度(m) vs 温度(℃)（色=湧出量）", 
    xlabel="深度(m)", ylabel="温度(℃)", xlim=(0,2000), ylim=(0,110) ,figsize=(14,8))

In [ ]:
scatter_series(df["温度上限"], x=df["pH上限"], c=df["湧出量"],cmap=plt.cm.Spectral, cmin=0, cmax=1000,
    size=80, colorbar=True, c_label="湧出量", title="pH vs 温度(℃)（色=湧出量）", alpha=0.5,
    xlabel="pH", ylabel="温度(℃)", ylim=(0,110) )

In [ ]:
sub = df[df["深度(m)"] < 5]
scatter_series(sub["温度上限"], x=sub["pH上限"], c=sub["湧出量"],cmap=plt.cm.Spectral, cmin=0, cmax=1000,
    annot=sub['名称'], size=80, colorbar=True, c_label="湧出量", title="pH vs 温度(℃)（色=湧出量）", 
    xlabel="pH", ylabel="温度(℃)", ylim=(0,110) )